In [24]:
import numpy as np
from numba import njit, experimental, int32
import numba as nb

In [89]:


spec = [
    ("batchsize", nb.int32),
    ("windowsize", nb.int32),
    ("lookforward", nb.int32),
    ("indexes", nb.int32[:]),
    # indexes where the pointers are
    ("tensors", nb.float32[:,:,:]),
    # data from tfdata, [timeframe, feature, entries]
    ("lookforwardoutput", nb.float32[:,:,:]),
    # lookforwardoutput, [batch, current + lookforward, labels]
    ("dataoutput", nb.float32[:,:,:,:]),
    # data output, [batch, timeframe, feature, entries]
    ("deltas", nb.float32[:]),
    # approximate time deltas for each timeframe
]

# Features are as follows: [timeframe, open, high, low, close, volume]

@experimental.jitclass(spec)

class slidecontainer:
    def __init__(self, tensors, batchsize, windowsize, lookforward):
        self.tensors = tensors
        """Indexed by increasing timeframe, base is at index 0 -> [timeframe, feature, entry]"""
        self.batchsize = batchsize
        self.windowsize = windowsize
        self.lookforward = lookforward
        self.indexes = np.zeros(shape=(tensors.shape[0]), dtype=np.int32)
        """Indexed by increasing timeframe, base is at index 0"""

        # Set up output buffers
        self.lookforwardoutput = np.zeros(shape=(self.batchsize, self.lookforward +1, 2), dtype=np.float32)
        self.dataoutput = np.zeros(shape=(self.batchsize, tensors.shape[0], tensors.shape[1], self.windowsize), dtype=np.float32)

        # Compute time deltas
        self.deltas = np.zeros(shape=(tensors.shape[0]), dtype=np.float32)
        for i in range(tensors.shape[0]):
            self.deltas[i] = self.tensors[i,0,1] - self.tensors[i,0,1]

    def getLatestTime(self) -> np.float32:
        latest = np.float32(0)
        for timeframe, index in enumerate(self.indexes):
            thislatest = self.tensors[timeframe][index]
            if thislatest > latest:
                latest = thislatest

        return latest

    def init_from_zero(self):
        # Slide all indexes to the minimum window size
        self.indexes[:] = self.windowsize

        latest = self.getLatestTime()
        self.stepToPresent(latest)


In [91]:
sc = slidecontainer(np.zeros(shape=(5, 5, 10000), dtype=np.float32), np.int32(50), np.int32(100), np.int32(5))

print(sc.deltas)

[0. 0. 0. 0. 0.]
